# Manufacturing Defect Detection — EDA & Model Evaluation
**Dataset:** MVTec Anomaly Detection  
**Model:** MobileNetV2 (Transfer Learning)  
**Author:** Arth Vichpuria


In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

sys.path.insert(0, '..')
plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#0a0c0f', 'axes.facecolor': '#111518',
                     'axes.edgecolor': '#1e2830', 'text.color': '#c8d6df',
                     'axes.labelcolor': '#c8d6df', 'xtick.color': '#c8d6df',
                     'ytick.color': '#c8d6df', 'grid.color': '#1e2830'})
ACCENT = '#00e5ff'
print('Imports OK | TF:', tf.__version__)

## 1. Dataset Exploration

In [ ]:
DATA_ROOT = Path('../data/mvtec')   # point to your processed dataset

def count_images(root):
    rows = []
    for split in ['train', 'val', 'test']:
        for cls in ['good', 'defective']:
            d = root / split / cls
            n = len(list(d.glob('*.png')) + list(d.glob('*.jpg'))) if d.exists() else 0
            rows.append({'split': split, 'class': cls, 'count': n})
    return pd.DataFrame(rows)

df_counts = count_images(DATA_ROOT)
print(df_counts.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
pivot = df_counts.pivot(index='split', columns='class', values='count')
pivot.plot(kind='bar', ax=ax, color=['#00e676', '#ff3d3d'], edgecolor='none', width=0.6)
ax.set_title('Dataset Split Distribution', pad=12)
ax.set_xlabel('Split'); ax.set_ylabel('Image Count')
ax.legend(title='Class')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Sample Images — Good vs Defective

In [ ]:
def show_samples(root, split='train', n=5):
    fig, axes = plt.subplots(2, n, figsize=(n * 3, 7))
    for col, cls in enumerate(['good', 'defective']):
        imgs = list((root / split / cls).glob('*.png'))[:n]
        for row, p in enumerate(imgs):
            img = Image.open(p).resize((224, 224))
            axes[col][row].imshow(img)
            axes[col][row].axis('off')
            if row == 0:
                axes[col][row].set_title(cls.upper(), color=ACCENT if cls=='good' else '#ff3d3d',
                                          fontsize=12, fontweight='bold')
    plt.suptitle('Sample Images — MVTec Dataset', fontsize=14, color='white')
    plt.tight_layout()
    plt.show()

show_samples(DATA_ROOT)

## 3. Pixel Intensity Distribution

In [ ]:
def pixel_histogram(root, split='train', n=50):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, cls, color in zip(axes, ['good', 'defective'], [ACCENT, '#ff3d3d']):
        imgs = list((root / split / cls).glob('*.png'))[:n]
        all_pixels = []
        for p in imgs:
            arr = np.array(Image.open(p).resize((64,64)).convert('L'))
            all_pixels.extend(arr.flatten().tolist())
        ax.hist(all_pixels, bins=64, color=color, alpha=0.8, edgecolor='none')
        ax.set_title(f'Pixel Distribution — {cls.upper()}')
        ax.set_xlabel('Pixel Value (0-255)')
        ax.set_ylabel('Frequency')
    plt.suptitle('Grayscale Intensity Distributions', fontsize=13)
    plt.tight_layout()
    plt.show()

pixel_histogram(DATA_ROOT)

## 4. Load Trained Model & Evaluate

In [ ]:
MODEL_PATH = '../models/defect_detection_mobilenetv2.keras'
model = tf.keras.models.load_model(MODEL_PATH)
print(f'Model loaded. Parameters: {model.count_params():,}')
model.summary()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    str(DATA_ROOT / 'test'),
    target_size=(224, 224), batch_size=32,
    class_mode='binary', shuffle=False
)

results = model.evaluate(test_gen, verbose=1)
for name, val in zip(model.metrics_names, results):
    print(f'  {name:12s}: {val:.4f}')

## 5. Confusion Matrix & Classification Report

In [ ]:
test_gen.reset()
y_pred_prob = model.predict(test_gen, verbose=1).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout(); plt.show()

## 6. ROC Curve & AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc     = auc(fpr, tpr)

prec, rec, _ = precision_recall_curve(y_true, y_pred_prob)
pr_auc       = auc(rec, prec)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color=ACCENT, lw=2, label=f'ROC AUC = {roc_auc:.3f}')
axes[0].plot([0,1],[0,1], '--', color='#4a6170', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend()

axes[1].plot(rec, prec, color='#00e676', lw=2, label=f'PR AUC = {pr_auc:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.suptitle('Model Performance Curves', fontsize=13)
plt.tight_layout(); plt.show()
print(f'ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}')

## 7. Grad-CAM on Test Images

In [ ]:
from utils.inference import generate_gradcam_b64, preprocess
import base64, io
from IPython.display import display, Image as IPImage

test_imgs = list((DATA_ROOT / 'test' / 'defective').glob('*.png'))[:3]
for img_path in test_imgs:
    batch  = preprocess(str(img_path))
    raw    = batch[0]
    b64    = generate_gradcam_b64(model, batch, raw)
    if b64:
        display(IPImage(data=base64.b64decode(b64), format='png'))
    print(f'  → {img_path.name}')